# CWL Execution - *CommandLineTool* Class 

Link to the documentation: https://eoap.github.io/quickwin/cwl-cli/water_bodies/

## CWL Inspection
In this Notebook we will inspect the CWL file saved in `../cwl-cli/detect-water-body.cwl`

In [ ]:
(
set -euo pipefail
cat ../cwl-cli/detect-water-body.cwl
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Let's inspect a few elements of this CWL CommandLineTool

What CWL class is it?

In [ ]:
(
set -euo pipefail
cat ../cwl-cli/detect-water-body.cwl | yq e .class -
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


What's its `id` ?

In [ ]:
(
set -euo pipefail
cat ../cwl-cli/detect-water-body.cwl | yq e .id -
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Inspect the base command and arguments:

In [ ]:
(
set -euo pipefail
cat ../cwl-cli/detect-water-body.cwl | yq e .baseCommand -
cat ../cwl-cli/detect-water-body.cwl | yq e .arguments -
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


This is equivalent to: 

```yaml
baseCommand: ["python"]
arguments: ["-m", "app"]
```

Let's inspect the `inputs` element:

In [ ]:
(
set -euo pipefail
cat ../cwl-cli/detect-water-body.cwl | yq e .inputs -
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


The `inputBinding` mapping provides a prefix to get to:

```
python -m app \
    --input-item <item input value> \
    --aoi <aoi input value> \
    --epsg <epsg input value> \
    --band <band input value [0]> \
    --band <band input value [1]> ...
```

Let's inspect the `outputs`.

Since the Python command line tool generates a STAC catalog, a STAC item and a geotiff, the glob expression will list all produced files in the folder the tool is run in.

In [ ]:
(
set -euo pipefail
cat ../cwl-cli/detect-water-body.cwl | yq e .outputs -
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Let's inspect the `requirements` element:

In [ ]:
(
set -euo pipefail
cat ../cwl-cli/detect-water-body.cwl | yq e .requirements -
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


The `EnvVarRequirement` adds an environment variable, an equivalent of:

```
export PYTHON=/app
```

The `ResourceRequirement` sets the computing resources requests, in this case one CPU and 512 Mib of RAM.

The `DockerRequirement` tels what container image to use, in this case `localhost/detect-water-body:latest`

**Final note:** all CWL documents must declare the CWL version used: 



In [ ]:
(
set -euo pipefail
cat ../cwl-cli/detect-water-body.cwl | yq e .cwlVersion -
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


## CWL Execution

Let's use a CWL runner to run this CWL document.

First create the `params.yaml` file:


In [ ]:
(
set -euo pipefail
cat << EOF > params.yaml
item:
  value: https://earth-search.aws.element84.com/v0/collections/sentinel-s2-l2a-cogs/items/S2B_10TFK_20210713_0_L2A
aoi:
  west: -121.399
  south: 39.834
  east: -120.74
  north: 40.472
epsg: "4326"
band:
- green
- nir
EOF
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Then use the `cwltool` to run the CWL document against the parameters:

In [ ]:
(
set -euo pipefail
cwltool --podman --outdir "${QUICKWIN_WORKSPACE:-/workspace/quickwin}/runs/cwl-cli" ../cwl-cli/detect-water-body.cwl params.yaml > cwl-cli-output.json
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Inspect the `cwl-cli-output.json` file keys:

In [ ]:
(
set -euo pipefail
cat cwl-cli-output.json | yq e '. | keys' - 
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


They're the same as: 

In [ ]:
(
set -euo pipefail
cat ../cwl-cli/detect-water-body.cwl | yq e .outputs -
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


Now get the root folder of the results:

In [ ]:
(
set -euo pipefail
cat cwl-cli-output.json | yq e '.water-body.location' -
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


We know there's a catalog.json file there:


In [ ]:
(
set -euo pipefail
stac describe $( cat cwl-cli-output.json | yq e '.water-body.location' - | sed 's|file://||g' )/catalog.json
)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi


In [ ]:
(
set -euo pipefail

)
status=$?
if [ "$status" -ne 0 ]; then echo "EOAP_CELL_FAILED:$status"; fi
